# Lab: Multi-Class Classification

*In this lab, we will build a multi-class classification model using scikit-learn, guided by the [Foundational Methodology for Data Science](../../../05_methodology/). While real-world projects require comprehensive documentation at each stage, this lab focuses on a streamlined, practical approach to demonstrate the core concepts and workflow. Therefore we will settle with the summaries of hypothetical stage reports.*

---

> ### 📝 1. Business Understanding Report (Summary)
> * **Business Context:** A botanical research institute has discovered that the *Iris-virginica* species possesses unique genetic properties that are valuable for developing new floral hybrids. The institute needs to collect pure samples of this species for their research.
> * **Business Problem:** The current process of identifying *Iris-virginica* relies on manual inspection, which is slow and costly. While [an initial proposal](../02_linear_and_logistic_regression/10_logistic_regression_lab.ipynb) was to build a simple binary classifier ("Is it Virginica?"), it has been determined that this is insufficient. A misclassification of another species (e.g., *Iris-setosa*) as *Iris-virginica* would contaminate the research samples, wasting significant time and resources. Therefore, the business needs a more precise system that can differentiate between all three species to ensure only pure *Iris-virginica* samples are selected.
> * **Project Goal:** To build a **multi-class classification model** that uses the physical measurements of an iris flower (sepal length, petal width, etc.) to automatically and accurately classify a specimen into one of the three distinct species: *Iris-setosa*, *Iris-versicolor*, or *Iris-virginica*.

---

> ### 📝 2. Analytic Approach Report (Summary)
> *   **Problem Type:** ***Supervised Multi-Class Classification***. The objective is no longer to predict a binary "yes/no" outcome but to assign each flower to one of three distinct and mutually exclusive categories (*Iris-setosa*, *Iris-versicolor*, or *Iris-virginica*).
> *   **Candidate Models:** To comprehensively evaluate the best approach for this new problem, we will implement and compare three distinct strategies:
>     1.  **Baseline (Native Multi-Class):** **Softmax Regression**. This is the natural extension of Logistic Regression for multi-class problems and will serve as our interpretable baseline.
>     2.  **Candidate 2 (Wrapper):** **One-vs-Rest (OvR) Classifier**. This strategy involves training one binary classifier for each class against all others.
>     3.  **Candidate 3 (Wrapper):** **One-vs-One (OvO) Classifier**. This strategy involves training a dedicated binary classifier for every possible pair of classes.
> *   **Evaluation Metrics:** Given the balanced nature of the Iris dataset and the need for high-confidence predictions, we will use a suite of metrics:
>     *   **Accuracy:** To provide a single, overall measure of the models' correctness.
>     *   **Confusion Matrix:** This is critical for visualizing the specific types of errors the model makes (e.g., how many *versicolor* samples are misclassified as *virginica*).
>     *   **Precision, Recall, and F1-Score (per class and weighted average):** To provide a nuanced evaluation of performance for each species and an aggregate score that accounts for class balance.
> *   **Validation Strategy:** The modeling process will involve splitting the data into training and testing sets (80/20 split) and applying **feature scaling** (`StandardScaler`), as the underlying linear models benefit from standardized features. The performance of all three candidate models will be compared on the held-out test set.

---

> ### 📝 3. Data Requirements Report (Summary)
> *   **Data Source:** The project will continue to use the classic `iris.csv` dataset, sourced from the Scikit-learn library (`sklearn.datasets.load_iris`), as established in the [previous lab](../02_linear_and_logistic_regression/10_logistic_regression_lab.ipynb). The dataset contains data on 150 iris flower samples.
> *   **Features (Independent Variables):** The features remain unchanged: `sepal_length`, `sepal_width`, `petal_length`, and `petal_width` (all measurements in centimeters).
> *   **Target (Dependent Variable):** This is the key change from the previous analysis. The target variable is now the original multi-class `species` column (or `target` from the scikit-learn object), which contains three distinct classes: *setosa* (0), *versicolor* (1), and *virginica* (2). We will no longer be engineering a binary `is_virginica` variable.
> *   **Granularity & Privacy:** The data granularity remains at the individual flower specimen level. The dataset is a well-known, public, and anonymized benchmark, containing no sensitive or personally identifiable information (PII), so there are no privacy concerns.

---

## Stage 4: Data Collection
As usual, we start with importing the necessary libraries and configuring them.

In [14]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier 
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

# Configurations
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")

### 4.1. Data Extraction and Storage
First, we Extract the data from its original source and store it in an untouched, "raw" format. This creates a perfect, versionable mirror of the source system at the time of collection.

> ⚠️ **Caution:** Even though we could use the already extracted data for our previous lab, this lab introduces a better Data Collection stage, including the ETL process. Therefore we'll extract the data from scratch.

In [15]:
raw_data = load_iris()
print(f"Data has been extracted from the source system as '{type(raw_data)}'")

Data has been extracted from the source system as '<class 'sklearn.utils._bunch.Bunch'>'


### 4.2. Data Transformation
Let's transform the data into a workable DataFrame and apply basic corrections.

In [16]:
# Create the DataFrame with the data and feature names
iris_interim = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)

# Add the target column
iris_interim['class'] = raw_data.target

# Add the string version of the target
iris_interim['class_name'] = raw_data.target_names[raw_data.target]

# Rename columns
iris_interim.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class', 'class_name']

# Sample the transformed DataFrame to verify
iris_interim.sample(5)

,sepal_length,sepal_width,petal_length,petal_width,class,class_name
37,4.9,3.6,1.4,0.1,0,setosa
95,5.7,3.0,4.2,1.2,1,versicolor
140,6.7,3.1,5.6,2.4,2,virginica
148,6.2,3.4,5.4,2.3,2,virginica
93,5.0,2.3,3.3,1.0,1,versicolor


In [22]:
# Check whether the data types are compatible and number of missing values don't indicate a data extraction issue
iris_interim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   class         150 non-null    int64  
 5   class_name    150 non-null    object 
dtypes: float64(4), int64(1), object(1)
memory usage: 7.2+ KB


### 4.3. Data Load
Let's now load the transformed data into its target destination.

In [23]:
# Define and create the interim data folder
interim_data_dir = Path("../data/interim")
interim_data_dir.mkdir(parents=True, exist_ok=True)

# Define the file path and save the DataFrame
interim_data_path = interim_data_dir / "iris_interim_v2.pkl"
iris_interim.to_pickle(interim_data_path)

### 4.4. Verification
Let's perform a final sanity check to verify that the data in the interim staging area is correct and ready to use.

In [24]:
iris_df = pd.read_pickle(interim_data_path)
iris_df.head()

,sepal_length,sepal_width,petal_length,petal_width,class,class_name
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


> ### 4. Data Collection Report (Summary)
> * 